# Lab 3.7 &mdash; Human-in-the-Loop

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; LangGraph: Stateful Agent Workflows**

### What you'll do
- Stop a graph before the step that cannot be taken back
- Resume it with <code>None</code> &mdash; and know why it is not the original input
- Let a person edit the state with <code>update_state</code>, then meet the rewind surprise

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All eight Module 3 labs work one case: leave requests in a small HR
> system. The rules are ordinary on purpose &mdash; the only new thing here is LangGraph.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-07")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# Leave requests in a small HR system. Ordinary rules on purpose: the only new thing in these
# eight labs is LangGraph. One flat dict -- no joins, no helpers, nothing to learn here.

REQUESTS = {
    "LV-5001": {"who": "Priya Nair",   "days":  3, "kind": "annual", "reason": "family wedding",
                "balance": 12, "manager": "Devi R."},
    "LV-5002": {"who": "Rahul Menon",  "days":  5, "kind": "annual", "reason": "",
                "balance":  3, "manager": "Devi R."},
    "LV-5003": {"who": "Anita Sharma", "days":  2, "kind": "annual", "reason": "moving house",
                "balance":  0, "manager": "Sam O."},
    "LV-5004": {"who": "Vikram Rao",   "days": 15, "kind": "annual", "reason": "sabbatical",
                "balance": 20, "manager": "Sam O."},
    "LV-5005": {"who": "Priya Nair",   "days":  1, "kind": "sick",   "reason": "flu",
                "balance": 12, "manager": "Devi R."},
}

# The handbook, as three numbers. Every routing decision in this module comes from these.
POLICY = {"manager_over_days": 2, "hr_over_days": 10, "max_clarifications": 2}

print(len(REQUESTS), "leave requests loaded")

## Concept

A checkpointer lets a run stop and start again. `interrupt_before` makes it stop on purpose.

```python
app = builder.compile(checkpointer=InMemorySaver(), interrupt_before=["notify"])
...
app.invoke(None, config)      # None means "carry on", not "start again"
```

`invoke()` runs up to `notify`, saves, and returns with `state.next == ("notify",)`. A person
looks. Then you resume.

**Where the gate goes is the design decision.** Not before the thinking &mdash; there is nothing to
look at yet, and you will have annoyed the approver by the third request. Before the step with a
consequence outside your process.

> An approval gate does not always need a checkpointer: a node that refuses to act without a named
> approver is already a gate, and the capstone uses exactly that. What a checkpointer buys is
> *pause and resume* &mdash; stopping now and finishing tomorrow, from another process.

## Section 1 &mdash; Where the gate belongs, and how to resume

Three nodes. One of them does something you cannot quietly undo.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

SENT = []          # stands in for the outside world: emails sent, balances debited


class ApprovalState(TypedDict):
    request_id: str
    recommendation: str
    approved_by: str
    notes: Annotated[list, add]


def recommend(state):
    """Works out what SHOULD happen. Nothing leaves the process."""
    r = REQUESTS[state["request_id"]]
    return {"recommendation": "granted" if r["balance"] >= r["days"] else "refused",
            "notes": ["recommend"]}

def notify(state):
    """Emails the employee and debits the balance. There is no unsend button."""
    SENT.append(f'{REQUESTS[state["request_id"]]["who"]}: leave {state["recommendation"]} '
                f'by {state["approved_by"] or "NOBODY"}')
    return {"notes": [f"notify -> {SENT[-1]}"]}


def gate_nodes() -> list:
    """Which node must not run until a person has said yes?"""
    # "recommend" -- computes a recommendation. Nothing leaves the process.
    # "notify"    -- emails the employee and debits their balance.
    return ["notify"]   # gate the irreversible step, not the thinking


def resume_input():
    """What do you hand invoke() to continue a paused run?"""
    start_again = {"request_id": "LV-5004", "notes": []}
    carry_on    = None
    return carry_on     # no new input -- resume from where you stopped


def build_gated():
    builder = StateGraph(ApprovalState)
    builder.add_node("recommend", recommend)
    builder.add_node("notify", notify)
    builder.add_edge(START, "recommend")
    builder.add_edge("recommend", "notify")
    builder.add_edge("notify", END)
    return builder.compile(checkpointer=InMemorySaver(), interrupt_before=gate_nodes())

In [ ]:
# --- Self-check: Section 1   (a real interrupt on a real graph -- no model)
BASE = {"recommendation": "", "approved_by": "", "notes": []}

def paused(rid="LV-5004"):
    SENT.clear()
    app, cfg = build_gated(), {"configurable": {"thread_id": rid}}
    app.invoke({**BASE, "request_id": rid}, cfg)
    return app, cfg

def paused_snapshot():
    app, cfg = paused()
    return app.get_state(cfg)

check("the run stops before notify, with the thinking already done",
      lambda: paused_snapshot().next == ("notify",)
          and paused_snapshot().values["recommendation"] == "granted",
      "state.next is what the graph would do if you let it carry on")
check("nothing reached the outside world while it was paused",
      lambda: (paused(), SENT == [])[1],
      "gating recommend instead would stop the run before there was anything to approve")

def resumed_values():
    app, cfg = paused()
    app.invoke(resume_input(), cfg)
    return app.get_state(cfg).values

check("resuming runs the gated node once, and does not re-run recommend",
      lambda: resumed_values()["notes"].count("recommend") == 1 and len(SENT) == 1,
      "passing the original input instead of None would run recommend a second time")
score()

## Section 2 &mdash; The person changes something

Approval is rarely just yes. `update_state` writes into the paused checkpoint before you resume,
which is how the approver's decision gets **into the record** rather than living in an email.

In [ ]:
def approver_writes(manager: str) -> dict:
    """A manager overrules the recommendation and refuses. What must land in the state?"""
    verdict_only    = {"recommendation": "refused"}
    verdict_and_who = {"recommendation": "refused", "approved_by": manager}
    return verdict_and_who   # a decision with no decider is not an audit record

In [ ]:
# --- Self-check: Section 2   (a human edit, then a real resume)
def overridden_values():
    app, cfg = paused()
    app.update_state(cfg, approver_writes("Sam O."))
    app.invoke(None, cfg)
    return app.get_state(cfg).values

check("the approver's verdict overrode the recommendation, and who decided is recorded",
      lambda: overridden_values()["recommendation"] == "refused"
          and overridden_values()["approved_by"] == "Sam O.",
      "the graph recommended granted; a person said refused, and the person won")
check("...and that is what reached the outside world",
      lambda: (overridden_values(), "refused" in SENT[-1])[1])
score()

## Watch it run

Run, pause, look, decide, resume.

In [ ]:
app = guard(build_gated)

if app is not None:
    SENT.clear()
    cfg = {"configurable": {"thread_id": "LV-5004"}}

    app.invoke({**BASE, "request_id": "LV-5004"}, cfg)
    s = app.get_state(cfg)
    print("1. paused before:", s.next, "  recommends:", s.values["recommendation"],
          "  sent so far:", SENT)

    app.update_state(cfg, {"approved_by": "Sam O.", "notes": ["Sam O. checked the team calendar"]})
    app.invoke(None, cfg)
    print("2. resumed, sent:", SENT)
    print("3. the record  :", app.get_state(cfg).values["notes"])

## Run it for real &mdash; and the surprise

The model drafts what the approver reads: a good use for it, because a person checks it before
anything happens. Then rewind into the gate.

In [ ]:
if llm_ready() and app is not None:
    SENT.clear()
    cfg2 = {"configurable": {"thread_id": "LV-5002-live"}}
    app.invoke({**BASE, "request_id": "LV-5002"}, cfg2)

    r = REQUESTS["LV-5002"]
    print("for the approver:", ask(
        f'Summarise for a manager deciding whether to approve: {r["who"]} asks for {r["days"]} '
        f'days of {r["kind"]} leave, reason given: "{r["reason"] or "none"}", balance '
        f'{r["balance"]} days. The system recommends '
        f'{app.get_state(cfg2).values["recommendation"]}.',
        system="You brief a busy manager. One sentence, no greeting.").strip()[:250])

    earlier = [h for h in app.get_state_history(cfg2) if h.next == ("recommend",)]
    if earlier:
        print("\nrewinding to before 'recommend' and running forward again...")
        app.invoke(None, earlier[0].config)
        print("   where did it stop?", app.get_state(cfg2).next, "  sent:", SENT)

### Read it

**The rewind stopped at the gate again** rather than running through to a new answer. That
surprises everyone once.

`interrupt_before` is a property of the **compiled graph**, not of a run. Replaying from an older
checkpoint runs forward through the same graph, so it meets the same gate &mdash; and rewinding
past the approval un-does it, because approval was a state edit at one point in history.

That is right for an approval workflow and wrong to discover in production. If you want a rewind
that keeps an approval, the approval has to be a **fact in the state** the gate reads, not the
fact that someone once resumed.

And `SENT` stayed empty until you resumed. The gate did its job.

In [ ]:
score()

## Your turn

1. Move the gate to `["recommend"]` and run it. What is the approver looking at? That is the
   argument for gating late.
2. An approver never comes back. Write the check that finds threads whose `next` has been
   non-empty for too long &mdash; timeout and escalation are policy on top of exactly this state.